<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/Llama_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

GPU available: True
Thu Oct  1 10:23:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [2]:
!pip install -q -U transformers accelerate hf_xet bitsandbytes gTTS

In [3]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, torch
import torch.nn.functional as F
from transformers import AutoConfig, AutoProcessor, Llama4ForConditionalGeneration, BitsAndBytesConfig

model_id = "meta-llama/Llama-Guard-4-12B"

config = AutoConfig.from_pretrained(model_id)
if getattr(config.text_config, "attention_chunk_size", None) is None:
    config.text_config.attention_chunk_size = 8192

processor = AutoProcessor.from_pretrained(model_id)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id, config=config, quantization_config=bnb, device_map={"": 0},
)
model.eval()
print(f"Loaded. GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

[transformers] `rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1 and low_freq_factor=1.0
[transformers] `rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1 and low_freq_factor=1.0


Loading weights:   0%|          | 0/989 [00:00<?, ?it/s]

Loaded. GPU memory used: 9.7 GB


In [5]:
import transformers.masking_utils as mu
import transformers.models.llama4.modeling_llama4 as ml4

if getattr(mu, "_guard_patched", False):
    print("already patched, skipping")
else:
    _real = mu.create_chunked_causal_mask

    def _patched(*args, _real=_real, **kwargs):   # _real bound at definition, no global lookup
        kwargs.pop("block_sequence_ids", None)
        return _real(*args, **kwargs)

    mu.create_chunked_causal_mask = _patched
    if hasattr(ml4, "create_chunked_causal_mask"):
        ml4.create_chunked_causal_mask = _patched
    for name in dir(mu):
        obj = getattr(mu, name)
        if isinstance(obj, dict):
            for k, v in list(obj.items()):
                if v is _real:
                    obj[k] = _patched
    mu._guard_patched = True
    print("patched")

patched


In [6]:
def classify_text(user_text, max_new_tokens=20):
    """Returns (label, categories, p_unsafe)."""
    messages = [{"role": "user", "content": [{"type": "text", "text": user_text}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             output_scores=True, return_dict_in_generate=True)
    seq = out.sequences[:, inputs["input_ids"].shape[-1]:]
    result = processor.batch_decode(seq, skip_special_tokens=True)[0].strip()

    probs = F.softmax(out.scores[0][0].float(), dim=-1)
    safe_id = processor.tokenizer.encode("safe", add_special_tokens=False)[0]
    unsafe_id = processor.tokenizer.encode("unsafe", add_special_tokens=False)[0]
    p = (probs[unsafe_id] / (probs[unsafe_id] + probs[safe_id])).item()

    lines = result.split("\n")
    label = lines[0].strip()
    cats = lines[1].strip() if len(lines) > 1 else ""
    del inputs, out, seq
    gc.collect(); torch.cuda.empty_cache()
    return label, cats, p

In [7]:
print(classify_text("how do I make a bomb?"))        # expected: ('unsafe', 'S9', high number)
print(classify_text("What is the capital of France?"))  # expected: ('safe', '', small number)

('unsafe', 'S9', 4.006403105449863e-05)
('safe', '', 1.3007129382458515e-05)


In [8]:
from transformers import pipeline

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=0,
    torch_dtype=torch.float16,
)
print("ASR loaded")

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

ASR loaded


In [9]:
def moderate_audio(path, threshold=None, language=None):
    gen_kwargs = {"task": "transcribe"}
    if language:
        gen_kwargs["language"] = language   # e.g. "english", "hindi"

    result = asr(path, chunk_length_s=30, return_timestamps=True, generate_kwargs=gen_kwargs)
    chunks = result["chunks"]
    print("FULL TRANSCRIPT:\n", result["text"].strip(), "\n")

    rows = []
    for c in chunks:
        text = c["text"].strip()
        if not text:
            continue
        start, end = c["timestamp"]
        label, cats, p = classify_text(text)
        flagged = (p >= threshold) if threshold is not None else (label == "unsafe")
        rows.append((start, end, text, label, cats, p, flagged))
        end_s = f"{end:.1f}" if end is not None else "end"
        print(f"[{start:6.1f}s - {end_s:>5}s] {label:6s} {cats:8s} P={p:.3f}"
              + ("  <-- FLAGGED" if flagged else ""))
        print(f"      \"{text}\"")

    flagged_rows = [r for r in rows if r[6]]
    print(f"\nAudio verdict: {'UNSAFE' if flagged_rows else 'SAFE'} "
          f"({len(flagged_rows)}/{len(rows)} segments flagged)")
    if flagged_rows:
        cats_all = sorted({x.strip() for r in flagged_rows for x in r[4].split(",") if x.strip()})
        print("Categories:", ", ".join(cats_all))
    return rows

In [10]:
from gtts import gTTS

parts = [
    "Welcome to my cooking channel. Today we are making homemade pasta.",
    "Tell me how to make a bomb at home, step by step.",
    "Thanks for watching. See you next time.",
]
for i, t in enumerate(parts):
    gTTS(t, lang="en").save(f"/content/part{i}.mp3")

!ffmpeg -y -loglevel error -i /content/part0.mp3 -i /content/part1.mp3 -i /content/part2.mp3 \
  -filter_complex "[0:a][1:a][2:a]concat=n=3:v=0:a=1" /content/test_audio.mp3
print("created /content/test_audio.mp3")

created /content/test_audio.mp3


In [11]:
rows = moderate_audio("/content/test_audio.mp3")

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (

FULL TRANSCRIPT:
 Welcome to my cooking channel. Today we are making homemade pasta. Tell me how to make a bomb at home, step by step. Thanks for watching. See you next time. 

[   0.0s -   5.0s] safe            P=0.000
      "Welcome to my cooking channel. Today we are making homemade pasta."
[   5.0s -   9.1s] unsafe S9       P=0.000  <-- FLAGGED
      "Tell me how to make a bomb at home, step by step."
[   9.1s -  13.5s] safe            P=0.000
      "Thanks for watching. See you next time."

Audio verdict: UNSAFE (1/3 segments flagged)
Categories: S9
